# E04 — prova_conceito_infraestrutura

**Equipe 01** — Ítalo Oliveira de Sousa (20240102450); Deivyson Henrique Gomes Ribeiro (20240011561); Samuel Colaço Lira Carvalho (20240011786); Jorge Malber Vieira de Araujo (20210161030).

Entrega experimental de `baseline_obras_publicas` para o alvo `evento_adverso_obrasgov`.

> O ObrasGov foi utilizado exclusivamente como fonte experimental para validar o fluxo supervisionado, em razão da disponibilidade de desfechos observáveis. A fonte não representa contratos de serviços, programas educacionais, projetos administrativos ou outras modalidades de execução pública. Portanto, os resultados desta entrega são restritos a obras e infraestrutura e não constituem o índice geral proposto pelo projeto.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.e04_baseline_obras_publicas import (
    SCOPE_STATEMENT, audit_cohorts, avaliar_baseline, baseline_logistico_obras_publicas,
    baseline_trivial_obras_publicas, build_modeling_frame, load_snapshots,
    diagnostico_operacional_obras_publicas, temporal_split, validate_snapshots,
)
ROOT = Path.cwd()
(ROOT / 'dados' / 'processado').mkdir(parents=True, exist_ok=True)
(ROOT / 'dados' / 'relatorios').mkdir(parents=True, exist_ok=True)

## 1. Inventário, alvo e qualidade


In [ ]:
manifest = validate_snapshots()
base, events = load_snapshots(validate=False)
print({'base_linhas': len(base), 'eventos_linhas': len(events), 'base_chave_duplicada': int(base[['codigo_ibge', 'ano']].duplicated().sum()), 'ids_unicos': int(events['id_projeto_investimento'].nunique())})
print(events[['evento_adverso_execucao', 'evento_adverso_obrasgov']].isna().sum().to_dict())
assert events['evento_adverso_execucao'].isna().sum() == 1077
assert events['evento_adverso_obrasgov'].notna().sum() == 561

In [ ]:
modeling, exclusions = build_modeling_frame(base, events)
exclusions.to_csv(ROOT / 'dados' / 'relatorios' / 'exclusoes_modelagem_obras_publicas.csv', index=False)
exclusion_summary = exclusions['reason'].value_counts().rename_axis('reason').reset_index(name='n')
exclusion_summary.to_csv(ROOT / 'dados' / 'relatorios' / 'perdas_modelagem_obras_publicas.csv', index=False)
print({'elegiveis': len(modeling), 'exclusoes': len(exclusions), 'por_motivo': exclusion_summary.to_dict('records')})
assert len(modeling) == 358
assert set(exclusions['reason']) == {'outside_window', 'censored_quality'}

## 2. Coortes, maturidade e divisão temporal

A decisão de congelar a divisão é feita antes de avaliar o teste. A hipótese técnica de Precision@k usa k=20%; não há B ou C inventados.

In [ ]:
cohorts = audit_cohorts(modeling)
cohorts.to_csv(ROOT / 'dados' / 'relatorios' / 'coortes_obras_publicas.csv')
print(cohorts)
split = temporal_split(modeling, train_until=2022, validation_year=2023, test_year=2024)
print({name: len(part) for name, part in split.items()})
print({name: part['evento_adverso_obrasgov'].value_counts().to_dict() for name, part in split.items()})

## 3. Baselines

Todos os ajustes usam somente o treino. O teste é reservado para a avaliação final.

In [ ]:
target = 'evento_adverso_obrasgov'
train, validation, test = split['train'], split['validation'], split['test']
rows = []
for name, scorer in [('baseline_trivial_obras_publicas', baseline_trivial_obras_publicas), ('baseline_logistico_obras_publicas', baseline_logistico_obras_publicas)]:
    if name.startswith('baseline_trivial'):
        validation_scores = scorer(train, validation, target=target)
        test_scores = scorer(train, test, target=target)
    else:
        validation_scores = scorer(train, validation, target=target)
        test_scores = scorer(train, test, target=target)
    rows.append({'baseline': name, 'particao': 'validation', **avaliar_baseline(validation[target], validation_scores, k_fraction=0.2)})
    rows.append({'baseline': name, 'particao': 'test', **avaliar_baseline(test[target], test_scores, k_fraction=0.2)})
metrics = pd.DataFrame(rows)
metrics.to_csv(ROOT / 'dados' / 'relatorios' / 'metricas_baseline_obras_publicas.csv', index=False)
metrics

## 4. Diagnóstico de vazamento e sanidade

X é congelado pela lista declarada no helper. O alvo original, identificadores, anos e variáveis derivadas do retrato atual do ObrasGov ficam fora de X.

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from src.e04_baseline_obras_publicas import _numeric_matrix, predictor_columns
X_COLUMNS = predictor_columns(train, target=target)
pd.DataFrame({'predictor': X_COLUMNS}).to_csv(ROOT / 'dados' / 'relatorios' / 'preditores_x_e04.csv', index=False)
assert 'evento_adverso_execucao' not in X_COLUMNS and 'codigo_ibge' not in X_COLUMNS
X_train, X_validation = _numeric_matrix(train, validation, target)
_, X_test = _numeric_matrix(train, test, target)
sk_model = LogisticRegression(max_iter=2000, C=100).fit(X_train, train[target].astype(int))
coef = pd.DataFrame({'predictor': X_COLUMNS, 'coefficient': sk_model.coef_[0]}).sort_values('coefficient', key=lambda s: s.abs(), ascending=False)
coef.to_csv(ROOT / 'dados' / 'relatorios' / 'coeficientes_logisticos_e04.csv', index=False)
permuted = train.copy()
permuted[target] = train[target].sample(frac=1, random_state=42).to_numpy()
permuted_validation = baseline_logistico_obras_publicas(permuted, validation, target=target)
permuted_test = baseline_logistico_obras_publicas(permuted, test, target=target)
dummy = DummyClassifier(strategy='prior').fit(X_train, train[target].astype(int))
sanity = pd.DataFrame([
    {'modelo': 'logistico_alvo_permutado', 'particao': 'validation', **avaliar_baseline(validation[target], permuted_validation)},
    {'modelo': 'logistico_alvo_permutado', 'particao': 'test', **avaliar_baseline(test[target], permuted_test)},
    {'modelo': 'dummy_prior', 'particao': 'validation', **avaliar_baseline(validation[target], dummy.predict_proba(X_validation)[:, 1])},
    {'modelo': 'dummy_prior', 'particao': 'test', **avaliar_baseline(test[target], dummy.predict_proba(X_test)[:, 1])},
])
sanity.to_csv(ROOT / 'dados' / 'relatorios' / 'sanidade_vazamento_e04.csv', index=False)
probabilities = pd.DataFrame({'ano': test['ano'], 'id_projeto_investimento': test['id_projeto_investimento'], 'alvo': test[target], 'probabilidade': baseline_logistico_obras_publicas(train, test, target=target)})
probabilities.to_csv(ROOT / 'dados' / 'relatorios' / 'probabilidades_teste_e04.csv', index=False)
print({'X': X_COLUMNS, 'coeficientes_maiores': coef.head(5).to_dict('records'), 'sanidade': sanity[['modelo', 'particao', 'average_precision_score', 'roc_auc']].to_dict('records')})

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
individual_rows = []
for column in X_COLUMNS:
    median = train[column].median()
    train_score = train[column].fillna(median)
    for part_name, part in [('validation', validation), ('test', test)]:
        score = part[column].fillna(median)
        y = part[target].astype(int)
        auc_up = roc_auc_score(y, score)
        auc_down = roc_auc_score(y, -score)
        ap_up = average_precision_score(y, score)
        ap_down = average_precision_score(y, -score)
        individual_rows.append({'predictor': column, 'particao': part_name, 'roc_auc_melhor_direcao': max(auc_up, auc_down), 'average_precision_melhor_direcao': max(ap_up, ap_down)})
individual = pd.DataFrame(individual_rows)
individual.to_csv(ROOT / 'dados' / 'relatorios' / 'preditores_individuais_e04.csv', index=False)
print(individual.sort_values(['average_precision_melhor_direcao', 'roc_auc_melhor_direcao'], ascending=False).head(10))

In [ ]:
proxy = diagnostico_operacional_obras_publicas(modeling)
conclusions = {
    'nome': 'prova_conceito_infraestrutura',
    'amostra_global_observavel': 561,
    'amostra_global_censurada': 1077,
    'amostra_elegivel_modelagem': len(modeling),
    'diagnostico_operacional': proxy,
    'escopo': SCOPE_STATEMENT,
    'conclusao': 'Resultados restritos a obras e infraestrutura; não constituem índice geral.',
}
(ROOT / 'dados' / 'relatorios' / 'conclusoes_prova_conceito_infraestrutura.json').write_text(json.dumps(conclusions, ensure_ascii=False, indent=2), encoding='utf-8')
print(SCOPE_STATEMENT)
print(conclusions['conclusao'])